In [1]:
import torch
from NeuralDecoder import MnistNeuralDecoder
import torch
from torch.utils.data import DataLoader , Dataset
from torch import nn
import torch.nn.functional as F
from torch.optim import Adam
from tqdm import tqdm
import copy
from GenerateNestedMnistData import CortexMnistDataset

In [2]:
folder='cortex_mnist'

In [3]:
import torch

def collate_fn(batch):
    spikes_on_list, spikes_off_list, images = zip(*batch)

    # create nested tensors
    spikes_on_nested = torch.nested.nested_tensor(spikes_on_list)
    spikes_off_nested = torch.nested.nested_tensor(spikes_off_list)

    # compute max time length
    t_max = max(s.shape[0] for s in spikes_on_list)

    # get other dims
    c, h, w = spikes_on_list[0].shape[1:]
    b = len(batch)

    # convert to padded tensors
    spikes_on_padded = torch.nested.to_padded_tensor(
        spikes_on_nested,
        padding=0.0,
        output_size=(b, t_max, c, h, w)
    )

    spikes_off_padded = torch.nested.to_padded_tensor(
        spikes_off_nested,
        padding=0.0,
        output_size=(b, t_max, c, h, w)
    )

    images = torch.stack(images)

    return spikes_on_padded, spikes_off_padded, images

In [4]:
train_dataset = CortexMnistDataset(f'{folder}/train.pt')
val_dataset   = CortexMnistDataset(f'{folder}/val.pt')
test_dataset  = CortexMnistDataset(f'{folder}/test.pt')

device=torch.device('mps')


In [5]:
son,soff,im=test_dataset[5]
print(im.dim())


3


In [6]:

model=MnistNeuralDecoder(grid_size=40)
son=son.unsqueeze(0)
son=son.unsqueeze(2).float()
print(son.shape)
out=model(son)

(11, 11)
torch.Size([1, 26, 1, 40, 40])


In [7]:
print(out.shape)

torch.Size([1, 1, 28, 28])


In [8]:
def train(model, num_epochs: int = 10):
    Epochs = num_epochs
    criterion = torch.nn.MSELoss()
    optimizer = Adam(params=model.parameters(), lr=5e-4)
    best_model = None
    best_loss = +torch.inf
    fake_batch_size = 50
    
    for epoch in range(Epochs):
        train_loss = 0.
        val_loss = 0.        
        model.train()
        train_batches = 0
        optimizer.zero_grad()
        for i, (spikes_on, _, images) in enumerate(tqdm(train_dataset, desc=f'Epoch {epoch+1}/{Epochs} Training')):
            spikes_on = spikes_on.float().to(device)
            images = images.float().to(device)
            spikes_on = spikes_on.unsqueeze(0)
            spikes_on = spikes_on.unsqueeze(2)
            out = model(spikes_on)
            out = out.squeeze(0)
            loss = criterion(out, images) 
            loss.backward()
            
            if (i + 1) % fake_batch_size == 0:
                optimizer.step()
                optimizer.zero_grad()
            
            train_loss += loss.item() 
            train_batches += 50
        
        # Step for any remaining samples that didn't complete a full fake batch
        if (i + 1) % fake_batch_size != 0:
            optimizer.step()
            optimizer.zero_grad()
        
        # Validation
        model.eval()
        val_batches = 0
        with torch.no_grad():
            for spikes_on, _, images in tqdm(test_dataset, desc=f'Epoch {epoch+1}/{Epochs} Validation'):
                spikes_on = spikes_on.float().to(device)
                images = images.float().to(device)
                spikes_on = spikes_on.unsqueeze(0)
                spikes_on = spikes_on.unsqueeze(2)
                out_val = model(spikes_on)
                out_val = out_val.squeeze(0)
                loss = criterion(out_val, images)
                
                val_loss += loss.item()
                val_batches += 50
        
        train_loss /= train_batches
        val_loss /= val_batches
        
        if val_loss < best_loss:
            print('New model saved')
            best_loss = val_loss
            best_model = copy.deepcopy(model.state_dict())
        
        print(f'Epoch {epoch+1}/{Epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
    
    return best_model

In [9]:



#def train(model, num_epochs: int = 5):
#    Epochs = num_epochs
#    criterion = torch.nn.MSELoss()
#    optimizer = Adam(params=model.parameters(), lr=1e-4)
#    best_model = None
#    best_loss = +torch.inf
#    
#    for epoch in range(Epochs):
#        train_loss = 0.
#        val_loss = 0.        
#        model.train()
#        train_batches = 0
#        for spikes_on, _, images in tqdm(train_dataset, desc=f'Epoch {epoch+1}/{Epochs} Training'):
#            optimizer.zero_grad()
#            spikes_on = spikes_on.float().to(device)
#            images = images.float().to(device)
#            spikes_on = spikes_on.unsqueeze(0)
#            spikes_on=spikes_on.unsqueeze(2)
#            out = model(spikes_on)
#            out = out.squeeze(0)
#            loss = criterion(out, images)
#            loss.backward()
#            optimizer.step()
#            
#            train_loss += loss.item()
#            train_batches += 1
#        
#        # Validation
#        model.eval()
#        val_batches = 0
#        with torch.no_grad():
#            for spikes_on, _, images in tqdm(test_dataset, desc=f'Epoch {epoch+1}/{Epochs} Validation'):
#                spikes_on = spikes_on.float().to(device)
#                images = images.float().to(device)
#                spikes_on = spikes_on.unsqueeze(0)
#                spikes_on=spikes_on.unsqueeze(2)
#                out_val = model(spikes_on)
#                out_val = out.squeeze(0)
#                loss = criterion(out_val, images)
#                
#                val_loss += loss.item()
#                val_batches += 1
#        
#        train_loss /= train_batches
#        val_loss /= val_batches
#        
#        if val_loss < best_loss:
#            print('New model saved')
#            best_loss = val_loss
#            best_model = copy.deepcopy(model.state_dict())
#        
#        print(f'Epoch {epoch+1}/{Epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}')
#    
#    return best_model

In [10]:
model=MnistNeuralDecoder(grid_size=40)
model.to(device)
best_model=train(model)

(11, 11)


Epoch 1/10 Validation: 100%|██████████| 500/500 [00:10<00:00, 47.71it/s]


New model saved
Epoch 1/10 | Train Loss: 0.0018 | Val Loss: 0.0016


Epoch 2/10 Validation: 100%|██████████| 500/500 [00:10<00:00, 45.74it/s]


New model saved
Epoch 2/10 | Train Loss: 0.0015 | Val Loss: 0.0015


Epoch 3/10 Training:  44%|████▍     | 2204/5000 [03:32<04:30, 10.35it/s]


KeyboardInterrupt: 

In [ ]:
torch.save(best_model,'NeuralDecoderNested.pth')

In [ ]:
model = MnistNeuralDecoder(grid_size=40)
model.load_state_dict(torch.load('NeuralDecoderNested.pth', map_location=device))  # ✅ Don't reassign
model = model.to(device)
model.eval()  # Good practice for inference
import random
idx = random.randint(0, len(test_dataset) - 1)
spikes_on, _, images = test_dataset[idx]
import matplotlib.pyplot as plt
spikes_on = spikes_on.float().to(device)
images = images.float().to(device)
spikes_on = spikes_on.unsqueeze(0)
spikes_on=spikes_on.unsqueeze(2)
out = model(spikes_on)
out = out.squeeze(0)
out = out.squeeze(0)


In [ ]:
plt.imshow(out.detach().cpu().numpy(),cmap='gray')
plt.colorbar()
plt.show()

In [ ]:
images=images.squeeze(0)
plt.imshow(images.detach().cpu().numpy(),cmap='gray')
plt.colorbar()
plt.show()